# 対象物アプローチの下降距離分析

pi05 Libero モデル（v44）の対象物直近での推論が、対象物との距離を正しく捉えているかを検証する。

処理の流れは次のとおり。

1. pick&place 系タスクを固定シードで実行し、推論ごとのシミュレータ状態とアクションチャンクを記録する
2. 推論時点の高さの差と、チャンクの下降距離を比較する
3. 異常な下降距離を出したロールアウトを抽出する
4. 抽出した推論を再現し、同じ入力で推論を繰り返して、サンプリングのばらつきと切り分ける

背景と仮説は `doc/theme.md` を参照。

## 1. セットアップ

VM の起動と実行の手順は `doc/colab_vm.md` を参照する。実行前に、次のファイルを VM にアップロードしておく。

| VM 上のパス | 内容 |
|---|---|
| `/content/rclone.conf` | Drive の認証情報。実行の最後に削除する |

Hugging Face トークンは、Drive の Google ドキュメント `pai_homework2/hf_token` から取得する。tokenizer の `google/paligemma-3b-pt-224` は利用申請が必要である。

Drive には `drive.mount` ではなく rclone でアクセスする。

In [ ]:
!pip install -q "lerobot[pi,libero]==0.6.2"
!curl -fsSL https://rclone.org/install.sh | bash > /dev/null

In [ ]:
import os
import pathlib
import subprocess

from huggingface_hub import login

# Hugging Face トークンを Drive から取得する
# Drive 上のファイルは Google ドキュメントなので、テキストとして書き出す。そのときの名前は hf_token になる
subprocess.run(
    ["rclone", "copyto", "remote:pai_homework2/hf_token", "/content/hf_token",
     "--drive-export-formats", "txt", "--config", "/content/rclone.conf"],
    check=True,
)
# 書き出したテキストの先頭には BOM が付くため、utf-8-sig で読む
login(token=pathlib.Path("/content/hf_token").read_text(encoding="utf-8-sig").strip())

# ヘッドレス環境で MuJoCo を GPU 描画する
os.environ["MUJOCO_GL"] = "egl"

# Colab の VM には NVIDIA の EGL 定義がない場合があり、CPU 描画に落ちて遅くなるため補う
icd = pathlib.Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

In [ ]:
import importlib.util

import yaml

# libero は設定ファイルがないと import 時に input() で止まるため、先に作っておく
libero_root = pathlib.Path(importlib.util.find_spec("libero.libero").origin).parent
config_file = pathlib.Path.home() / ".libero/config.yaml"
config_file.parent.mkdir(exist_ok=True)
config_file.write_text(yaml.dump({
    "benchmark_root": str(libero_root),
    "bddl_files": str(libero_root / "bddl_files"),
    "init_states": str(libero_root / "init_files"),
    "datasets": str(libero_root.parent / "datasets"),
    "assets": str(libero_root / "assets"),
}))

## 2. 設定

`SUITES` は、全タスクが pick&place である `libero_spatial` と `libero_object` にしている。

`REPLAN_STEPS` が推論間隔で、対象物の直近でも推論が入るよう短めにしている。

In [ ]:
MODEL_ID = "lerobot/pi05_libero_finetuned_v044"
SUITES = ["libero_spatial", "libero_object"]
EPISODES_PER_TASK = 5
REPLAN_STEPS = 5
SEED = 7

# 環境の設定。モデルの入力解像度に合わせる
IMAGE_SIZE = 256
SETTLE_STEPS = 10  # 開始直後に物体が落ち着くのを待つステップ数
MAX_STEPS = {"libero_spatial": 280, "libero_object": 280}
DUMMY_ACTION = [0, 0, 0, 0, 0, 0, -1]  # 動かず、グリッパーは開いたまま

# 分析の設定
ERROR_THRESHOLD_M = 0.02  # 異常とみなす下降距離の誤差。暫定値
TOP_N = 5  # 繰り返し推論を行うロールアウト数
REPEATS = 16  # 1 つの状態に対する繰り返し推論の回数

# データの保存先
LOCAL_DIR = pathlib.Path("/content/results")
EPISODE_DIR = LOCAL_DIR / "episodes"
ARCHIVE = pathlib.Path("/content/results.tar.gz")
REMOTE_ARCHIVE = "remote:pai_homework2/approach_analysis/results.tar.gz"
RCLONE_CONF = "/content/rclone.conf"
EPISODE_DIR.mkdir(parents=True, exist_ok=True)

## 3. モデルと環境

推論では `predict_action_chunk` を直接呼び、チャンク全体を受け取る。

初期ノイズは `(シード, スイート, タスク, エピソード, ステップ, 試行番号)` から決める。これにより、推論の順序や回数に関係なく、同じ推論では同じノイズを再現できる。試行番号を変えれば、乱数だけを変えて繰り返し推論できる。

In [ ]:
import zlib

import numpy as np
import torch
from lerobot.configs.policies import PreTrainedConfig
from lerobot.envs.libero import get_task_init_states
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
from libero.libero import benchmark, get_libero_path
from libero.libero.envs import OffScreenRenderEnv

config = PreTrainedConfig.from_pretrained(MODEL_ID)
config.compile_model = False  # コンパイルは起動に時間がかかり、検証には不要
policy = PI05Policy.from_pretrained(MODEL_ID, config=config).eval()
preprocess, postprocess = make_pre_post_processors(config, pretrained_path=MODEL_ID)

suite_tasks = {name: benchmark.get_benchmark_dict()[name]() for name in SUITES}

In [ ]:
def make_env(suite_name, task_id):
    """タスクの環境、初期状態、指示文、把持対象の名前を返す。"""
    suite = suite_tasks[suite_name]
    task = suite.get_task(task_id)
    bddl_file = pathlib.Path(get_libero_path("bddl_files")) / task.problem_folder / task.bddl_file
    env = OffScreenRenderEnv(bddl_file_name=str(bddl_file), camera_heights=IMAGE_SIZE, camera_widths=IMAGE_SIZE)
    # 対象の 2 スイートでは、ゴール条件の第 1 引数が把持対象になる。例: (On akita_black_bowl_1 plate_1)
    target = env.env.parsed_problem["goal_state"][0][1]
    return env, get_task_init_states(suite, task_id), task.language, target


def quat_to_axis_angle(quat):
    """(x, y, z, w) のクォータニオンを軸角に変換する。"""
    w = np.clip(quat[3], -1.0, 1.0)
    sin_half = np.sqrt(1.0 - w * w)
    if sin_half < 1e-10:
        return np.zeros(3)
    return quat[:3] / sin_half * 2.0 * np.arccos(w)


def image_tensor(image):
    """学習データの向きに合わせて 180 度回転し、[0, 1] の (1, C, H, W) に変換する。"""
    return torch.from_numpy(image[::-1, ::-1].copy()).permute(2, 0, 1)[None].float() / 255


def infer_chunk(obs, prompt, noise_key):
    """観測から、正規化を解除したアクションチャンク (chunk_size, 7) を返す。"""
    state = np.concatenate([obs["robot0_eef_pos"], quat_to_axis_angle(obs["robot0_eef_quat"]), obs["robot0_gripper_qpos"]])
    batch = {
        "observation.images.image": image_tensor(obs["agentview_image"]),
        "observation.images.image2": image_tensor(obs["robot0_eye_in_hand_image"]),
        "observation.state": torch.from_numpy(state)[None].float(),
        "task": [prompt],
    }
    generator = torch.Generator().manual_seed(zlib.crc32(noise_key.encode()))
    noise = torch.randn(1, config.chunk_size, config.max_action_dim, generator=generator)
    with torch.inference_mode():
        chunk = postprocess(policy.predict_action_chunk(preprocess(batch), noise=noise))
    return chunk[0].float().cpu().numpy()

## 4. データ収集

1 エピソードごとに、次のデータを `.npz` で保存する。

| キー | 内容 |
|---|---|
| `infer_t` | 推論したステップ |
| `infer_state` | 推論時のシミュレータ状態 |
| `chunks` | 推論で出たアクションチャンク全体 |
| `step_action` | 実際に実行したアクション |
| `step_eef` / `step_obj` | 各ステップの手先位置と対象物位置 |
| `success` | タスクの成否 |

`run_episode` に `stop_at` を渡すと、そのステップで止まって観測を返す。第 7 章の再現で使う。

結果はファイル数が多いため、1 個のアーカイブにまとめて Drive に送る。タスクごとに送り直すので、VM が回収されても新しい VM で実行し直せば続きから収集できる。

In [ ]:
def run_episode(env, init_states, prompt, target, suite_name, task_id, episode, stop_at=None):
    """1 エピソードを実行し、記録を返す。stop_at を指定すると、そのステップの観測を返す。"""
    env.seed(SEED)
    env.reset()
    obs = env.set_init_state(init_states[episode % len(init_states)])
    for _ in range(SETTLE_STEPS):
        obs, _, _, _ = env.step(DUMMY_ACTION)

    sim = env.env.sim
    target_body = env.env.obj_body_id[target]
    log = {key: [] for key in ["infer_t", "infer_state", "chunks", "step_action", "step_eef", "step_obj"]}
    success = False

    for t in range(MAX_STEPS[suite_name]):
        if t == stop_at:
            return obs
        if t % REPLAN_STEPS == 0:
            chunk = infer_chunk(obs, prompt, f"{SEED}/{suite_name}/{task_id}/{episode}/{t}/0")
            log["infer_t"].append(t)
            log["infer_state"].append(env.get_sim_state())
            log["chunks"].append(chunk)

        # 環境の入力範囲に収める
        action = np.clip(chunk[t % REPLAN_STEPS], -1.0, 1.0)
        log["step_action"].append(action)
        log["step_eef"].append(obs["robot0_eef_pos"].copy())
        log["step_obj"].append(sim.data.body_xpos[target_body].copy())

        obs, _, done, _ = env.step(action)
        if done:
            success = True
            break

    return {key: np.asarray(value) for key, value in log.items()} | {"success": success}

In [ ]:
def sync_to_drive():
    """結果をアーカイブにまとめて Drive に送る。"""
    subprocess.run(["tar", "czf", str(ARCHIVE), "-C", str(LOCAL_DIR.parent), LOCAL_DIR.name], check=True)
    subprocess.run(["rclone", "copyto", str(ARCHIVE), REMOTE_ARCHIVE, "--config", RCLONE_CONF], check=True)


# Drive に前回の結果があれば取得する。初回は存在しないので失敗してよい
subprocess.run(["rclone", "copyto", REMOTE_ARCHIVE, str(ARCHIVE), "--config", RCLONE_CONF], check=False)
if ARCHIVE.exists():
    subprocess.run(["tar", "xzf", str(ARCHIVE), "-C", str(LOCAL_DIR.parent)], check=True)

for suite_name in SUITES:
    for task_id in range(suite_tasks[suite_name].n_tasks):
        env, init_states, prompt, target = make_env(suite_name, task_id)
        for episode in range(EPISODES_PER_TASK):
            path = EPISODE_DIR / f"{suite_name}_task{task_id:02d}_ep{episode:03d}.npz"
            if path.exists():
                continue
            result = run_episode(env, init_states, prompt, target, suite_name, task_id, episode)
            np.savez_compressed(path, suite=suite_name, task_id=task_id, episode=episode, target=target, **result)
            print(path.name, "success" if result["success"] else "failure", flush=True)
        env.close()
        sync_to_drive()

## 5. 下降距離の分析

エピソード内で最初にグリッパーを閉じたステップを、把持の開始とみなす。それより前の推論だけを分析に使うので、配置先への下降は混ざらない。

### アクションからメートルへの換算

推論間隔の間に実際に動いた高さと、指令した z アクションの合計を最小二乗法で当てはめ、換算係数を実測する。

### 必要な下降距離

対象物の形状によって把持する高さが違うため、成功エピソードの実績から求める。

- `把持オフセット` = 把持開始時の `手先の高さ - 対象物の高さ` の、タスクごとの中央値
- `必要な下降距離` = `手先の高さ - 対象物の高さ - 把持オフセット`

### チャンクの下降距離と誤差

- `計画下降距離` = チャンク内で最初にグリッパーを閉じるまでの、z アクションの合計を換算した値
- `誤差` = `計画下降距離 - 必要な下降距離`。正なら突っ込みすぎ、負なら届かない

把持まで計画していないチャンクは下降距離を比較できないため、グリッパーを閉じる計画を含むチャンクだけを使う。

In [ ]:
import pandas as pd


def first_close(actions):
    """グリッパーを最初に閉じるインデックスを返す。閉じなければ None を返す。"""
    closing = np.flatnonzero(actions[:, 6] > 0)
    return int(closing[0]) if len(closing) else None


def load_episodes():
    episodes = []
    for path in sorted(EPISODE_DIR.glob("*.npz")):
        with np.load(path) as data:
            episode = {key: data[key] for key in data.files}
        for key in ["suite", "target"]:
            episode[key] = str(episode[key])
        for key in ["task_id", "episode"]:
            episode[key] = int(episode[key])
        episode["success"] = bool(episode["success"])
        episode["close_t"] = first_close(episode["step_action"])
        episodes.append(episode)
    return episodes


def estimate_gain(episodes):
    """z アクションの合計に対する、実際の高さ変化の比を返す。"""
    commanded, moved = [], []
    for ep in episodes:
        end = ep["close_t"] if ep["close_t"] is not None else len(ep["step_action"])
        for t in ep["infer_t"]:
            if t + REPLAN_STEPS < end:
                commanded.append(ep["step_action"][t:t + REPLAN_STEPS, 2].sum())
                moved.append(ep["step_eef"][t + REPLAN_STEPS, 2] - ep["step_eef"][t, 2])
    commanded, moved = np.array(commanded), np.array(moved)
    gain = commanded @ moved / (commanded @ commanded)
    r2 = np.corrcoef(commanded, moved)[0, 1] ** 2
    return gain, r2


def grasp_offsets(episodes):
    """タスクごとの把持オフセットを返す。"""
    rows = [
        {"suite": ep["suite"], "task_id": ep["task_id"],
         "offset": ep["step_eef"][ep["close_t"], 2] - ep["step_obj"][ep["close_t"], 2]}
        for ep in episodes if ep["success"] and ep["close_t"] is not None
    ]
    return pd.DataFrame(rows).groupby(["suite", "task_id"])["offset"].median().to_dict()


def planned_descent(chunk, gain):
    """グリッパーを閉じるまでの下降距離を返す。閉じる計画がなければ None を返す。"""
    close = first_close(chunk)
    if close is None:
        return None
    return -gain * np.clip(chunk[:close, 2], -1.0, 1.0).sum()


def build_table(episodes, gain, offsets):
    """把持開始より前の推論を、1 推論 1 行の表にする。"""
    rows = []
    for ep in episodes:
        offset = offsets.get((ep["suite"], ep["task_id"]))
        if offset is None or ep["close_t"] is None:
            continue
        # 把持開始時に、手先が基準の高さからどれだけずれていたか。正なら高い位置で閉じた
        close_gap = ep["step_eef"][ep["close_t"], 2] - ep["step_obj"][ep["close_t"], 2] - offset
        for index, t in enumerate(ep["infer_t"]):
            planned = planned_descent(ep["chunks"][index], gain)
            if t >= ep["close_t"] or planned is None:
                continue
            required = ep["step_eef"][t, 2] - ep["step_obj"][t, 2] - offset
            rows.append({
                "suite": ep["suite"], "task_id": ep["task_id"], "episode": ep["episode"], "t": int(t),
                "success": ep["success"], "required": required, "planned": planned,
                "error": planned - required, "close_gap": close_gap,
            })
    return pd.DataFrame(rows)


episodes = load_episodes()
gain, r2 = estimate_gain(episodes)
offsets = grasp_offsets(episodes)
table = build_table(episodes, gain, offsets)

print(f"エピソード数: {len(episodes)}, 成功率: {np.mean([ep['success'] for ep in episodes]):.1%}")
print(f"換算係数: {gain:.4f} m/アクション (R2={r2:.3f})")
print(f"分析対象の推論数: {len(table)}")
print(f"成功例がなく除外したタスク: {sorted({(ep['suite'], ep['task_id']) for ep in episodes} - set(offsets))}")

### 高さの差ごとの誤差

仮説が正しければ、必要な下降距離が小さい（対象物に近い）区間ほど、誤差の広がりが大きくなるはずである。

In [ ]:
import matplotlib.pyplot as plt

HEIGHT_BINS_M = [-np.inf, 0.01, 0.02, 0.04, 0.06, 0.08, 0.12, np.inf]

table["height_bin"] = pd.cut(table["required"], HEIGHT_BINS_M)
summary = table.groupby("height_bin", observed=True)["error"].agg(
    count="count", mean="mean", std="std",
    p5=lambda x: x.quantile(0.05), p95=lambda x: x.quantile(0.95),
)
summary["abnormal_rate"] = table.groupby("height_bin", observed=True)["error"].apply(
    lambda x: (x.abs() > ERROR_THRESHOLD_M).mean()
)
print(summary.round(4))
summary.to_csv(LOCAL_DIR / "error_by_height.csv")

fig, ax = plt.subplots(figsize=(7, 4))
for success, marker, label in [(True, "o", "success"), (False, "x", "failure")]:
    subset = table[table["success"] == success]
    ax.scatter(subset["required"] * 100, subset["error"] * 100, s=10, marker=marker, alpha=0.5, label=label)
ax.axhline(0, color="gray", linewidth=1)
ax.set_xlabel("Required descent [cm]")
ax.set_ylabel("Planned - required [cm]  (+: overshoot, -: undershoot)")
ax.legend()
fig.tight_layout()
fig.savefig(LOCAL_DIR / "error_by_height.png", dpi=150)
plt.show()

## 6. 問題ロールアウトの抽出

誤差の絶対値が `ERROR_THRESHOLD_M` を超えた推論を異常とみなし、エピソードごとに誤差が最大の推論を 1 つ選ぶ。

`close_gap` は、実際にグリッパーを閉じた高さの基準からのずれである。正なら届かないまま閉じた、負なら基準より深く入ったことを示す。

In [ ]:
abnormal = table[table["error"].abs() > ERROR_THRESHOLD_M]
worst_index = abnormal["error"].abs().groupby([abnormal["suite"], abnormal["task_id"], abnormal["episode"]]).idxmax()
problems = table.loc[worst_index].drop(columns="height_bin")
problems = problems.sort_values("error", key=np.abs, ascending=False).reset_index(drop=True)

problems.to_csv(LOCAL_DIR / "problem_rollouts.csv", index=False)
print(f"異常な推論を含むエピソード: {len(problems)} / {len(episodes)}")
problems.head(20).round(4)

## 7. 問題ロールアウトの再現と繰り返し推論

`problems` の上位について、固定シードで該当ステップまで再現する。再現した状態が記録と一致することを確認してから、シミュレータを進めずに乱数だけを変えて推論を繰り返す。

`repeat_mean` の誤差も閾値を超えていれば、たまたまの外れ値ではなく、その状態で一貫して距離を誤っていると判断できる。

In [ ]:
def repeat_inference(row):
    """問題の推論を再現し、REPEATS 回の推論の下降距離を返す。"""
    suite_name, task_id, episode, t = row.suite, row.task_id, row.episode, row.t
    env, init_states, prompt, target = make_env(suite_name, task_id)
    obs = run_episode(env, init_states, prompt, target, suite_name, task_id, episode, stop_at=t)

    # 再現した状態を記録と照合する
    path = EPISODE_DIR / f"{suite_name}_task{task_id:02d}_ep{episode:03d}.npz"
    with np.load(path) as data:
        recorded = data["infer_state"][list(data["infer_t"]).index(t)]
    reproduced = np.allclose(env.get_sim_state(), recorded, atol=1e-6)

    descents = []
    for repeat in range(1, REPEATS + 1):
        chunk = infer_chunk(obs, prompt, f"{SEED}/{suite_name}/{task_id}/{episode}/{t}/{repeat}")
        descents.append(planned_descent(chunk, gain))
    env.close()
    # 閉じる計画がなかった試行は比較できないため除く
    return reproduced, np.array([d for d in descents if d is not None])


rows = []
for row in problems.head(TOP_N).itertuples():
    reproduced, descents = repeat_inference(row)
    if not reproduced:
        print(f"状態が一致しないためスキップ: {row.suite} task{row.task_id} ep{row.episode} t={row.t}")
        continue
    repeat_error = descents - row.required
    rows.append({
        "suite": row.suite, "task_id": row.task_id, "episode": row.episode, "t": row.t,
        "required": row.required, "original_error": row.error, "valid_repeats": len(descents),
        "repeat_mean": repeat_error.mean(), "repeat_std": repeat_error.std(),
        "consistent": abs(repeat_error.mean()) > ERROR_THRESHOLD_M,
    })

repeats = pd.DataFrame(rows)
repeats.to_csv(LOCAL_DIR / "repeat_inference.csv", index=False)
sync_to_drive()
repeats.round(4)

## 8. 後始末

Drive の認証情報と Hugging Face トークンを VM から削除する。VM の停止は手元から `colab stop` で行う。

In [ ]:
!shred -u /content/rclone.conf /content/hf_token